In [1]:
!pip install -q segmentation-models-pytorch albumentations scikit-image networkx scipy opencv-python-headless pandas Pillow torch torchvision
import torch
print('CUDA:', torch.cuda.is_available())
print('torch:', torch.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 1.4 MB/s eta 0:00:00
CUDA: False
torch: 2.11.0+cpu


In [11]:
import os
for d in [
    '/content/root_project/data/images',
    '/content/root_project/data/masks',
    '/content/root_project/data/rsml',
    '/content/root_project/data/test_images',
    '/content/root_project/checkpoints',
    '/content/root_project/outputs/masks',
    '/content/root_project/outputs/skeletons',
    '/content/root_project/outputs/results',
    '/content/root_project/src',
]:
    os.makedirs(d, exist_ok=True)
print('Folders created.')

Folders created.


In [4]:

IMG_SIZE  = 768
THRESHOLD = 0.5
print(f"IMG_SIZE = {IMG_SIZE}   THRESHOLD = {THRESHOLD}")

IMG_SIZE = 768   THRESHOLD = 0.5


In [5]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

images_dir = "/content/root_project/data/images"
masks_dir  = "/content/root_project/data/masks"

files = sorted([
    f for f in os.listdir(images_dir)
    if f.lower().endswith((".png",".jpg",".jpeg"))
    and os.path.exists(os.path.join(masks_dir, f))
])

print(f"Total pairs: {len(files)}")
print()

for fname in files[:5]:
    img  = cv2.imread(os.path.join(images_dir, fname))
    mask = cv2.imread(os.path.join(masks_dir,  fname), cv2.IMREAD_GRAYSCALE)

    img_h,  img_w  = img.shape[:2]
    msk_h,  msk_w  = mask.shape[:2]
    pos_px         = (mask > 127).sum()
    total_px       = mask.size
    coverage       = pos_px / total_px * 100

    flag = "  <-- ROTATED (will be auto-fixed in dataset.py)" if (img_h == msk_w and img_w == msk_h and img_h != msk_h) else ""
    print(f"{fname}")
    print(f"  Image : {img_w}x{img_h}")
    print(f"  Mask  : {msk_w}x{msk_h}{flag}")
    print(f"  Root pixels: {pos_px} / {total_px} = {coverage:.4f}%")
    print()

fname = files[0]
img   = cv2.cvtColor(cv2.imread(os.path.join(images_dir, fname)), cv2.COLOR_BGR2RGB)
mask  = cv2.imread(os.path.join(masks_dir, fname), cv2.IMREAD_GRAYSCALE)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].imshow(img);  ax[0].set_title(f'Image: {fname}')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title('Mask')
plt.tight_layout()
plt.savefig('/content/root_project/outputs/data_check.png', dpi=80)
plt.show()
print("Saved data_check.png")

Total pairs: 0



IndexError: list index out of range

In [7]:

for mod in list(__import__('sys').modules.keys()):
    if 'dataset' in mod:
        del __import__('sys').modules[mod]

with open('/content/root_project/src/dataset.py', 'w') as f:
    f.write('''
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import albumentations as A

class RootDataset(Dataset):
    def __init__(self, images_dir, masks_dir, file_list, img_size=768, train=True):
        self.images_dir = images_dir
        self.masks_dir  = masks_dir
        self.file_list  = file_list
        self.img_size   = img_size
        self.train      = train

        if train:
            self.transforms = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
                A.GaussianBlur(blur_limit=(3,5), p=0.2),
                A.RandomGamma(gamma_limit=(80,120), p=0.3),
            ], is_check_shapes=False)
        else:
            self.transforms = A.Compose([
                A.Resize(img_size, img_size),
            ], is_check_shapes=False)

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        fname = self.file_list[idx]

        image = cv2.imread(os.path.join(self.images_dir, fname))
        if image is None:
            raise FileNotFoundError(f"Image not found: {fname}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.masks_dir, fname), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask not found: {fname}")

        img_h, img_w = image.shape[:2]
        msk_h, msk_w = mask.shape[:2]

        # Fix rotated masks
        if img_h == msk_w and img_w == msk_h and img_h != msk_h:
            mask = cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
            msk_h, msk_w = mask.shape[:2]

        if msk_h != img_h or msk_w != img_w:
            mask = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)

        # ── DILATION: this is the key fix ─────────────────────────────────
        # Your original masks have 1-3px wide root lines drawn at 4096x2304.
        # After downscaling to 768x768 (factor ~5-6x), those lines become
        # sub-pixel and disappear completely. The model then never sees actual
        # thin-line targets during training, so it learns to predict blurry
        # blobs instead.
        #
        # We dilate the mask BEFORE downscaling so the lines survive the resize.
        # The dilation kernel of 9px at 4096x2304 becomes ~1.5px after resize —
        # just enough to be a visible, learnable target.
        #
        # At inference time you already skeletonize the predicted mask back to
        # a thin line, so dilation here does NOT inflate your TRL measurement.
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
        mask   = cv2.dilate(mask, kernel, iterations=1)
        # ──────────────────────────────────────────────────────────────────

        mask = (mask > 127).astype(np.float32)

        aug   = self.transforms(image=image, mask=mask)
        image = aug["image"].astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        mask  = np.expand_dims(aug["mask"], axis=0)

        return torch.tensor(image), torch.tensor(mask)
''')
print("dataset.py rewritten with mask dilation fix")

# Verify dilation actually produces visible masks at training resolution
import cv2, numpy as np, os, matplotlib.pyplot as plt

images_dir = "/content/root_project/data/images"
masks_dir  = "/content/root_project/data/masks"
fname = sorted(os.listdir(masks_dir))[0]

mask_orig = cv2.imread(os.path.join(masks_dir, fname), cv2.IMREAD_GRAYSCALE)
h, w = mask_orig.shape[:2]

kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9,9))
mask_dil = cv2.dilate(mask_orig, kernel, iterations=1)

# Resize both to training size to show what the model actually sees
mask_orig_small = cv2.resize(mask_orig, (768,768), interpolation=cv2.INTER_NEAREST)
mask_dil_small  = cv2.resize(mask_dil,  (768,768), interpolation=cv2.INTER_NEAREST)

orig_px = (mask_orig_small > 127).sum()
dil_px  = (mask_dil_small  > 127).sum()

fig, axes = plt.subplots(1,4,figsize=(20,5))
axes[0].imshow(mask_orig, cmap='gray'); axes[0].set_title(f'Original mask\n{mask_orig.shape}')
axes[1].imshow(mask_dil,  cmap='gray'); axes[1].set_title(f'Dilated mask (9px kernel)\n{mask_dil.shape}')
axes[2].imshow(mask_orig_small, cmap='gray'); axes[2].set_title(f'Original at 768x768\n{orig_px} root px')
axes[3].imshow(mask_dil_small,  cmap='gray'); axes[3].set_title(f'Dilated at 768x768\n{dil_px} root px')
plt.tight_layout()
plt.savefig('/content/root_project/outputs/dilation_check.png', dpi=80)
plt.show()

print(f"\nOriginal mask at 768x768: {orig_px} root pixels ({orig_px/768**2*100:.3f}%)")
print(f"Dilated  mask at 768x768: {dil_px} root pixels ({dil_px/768**2*100:.3f}%)")
print("\nIf 'Dilated at 768x768' shows clear white lines -> dilation fix is working")
print("If original shows almost nothing -> this confirms why the model was learning blobs")

dataset.py rewritten with mask dilation fix


IndexError: list index out of range

In [25]:
import os
imgs  = set(os.listdir('/content/root_project/data/images'))
masks = set(os.listdir('/content/root_project/data/masks'))
matched = imgs & masks
missing = imgs - masks
print(f"Matched pairs : {len(matched)}")
if missing:
    print(f"Images missing a mask: {sorted(missing)}")
else:
    print("All images have matching masks.")

Matched pairs : 115
Images missing a mask: ['.ipynb_checkpoints']


In [11]:
import sys
sys.path.insert(0, '/content/root_project/src')
if 'dataset' in sys.modules: del sys.modules['dataset']

import os
from dataset import RootDataset

images_dir = "/content/root_project/data/images"
masks_dir  = "/content/root_project/data/masks"

files = sorted([
    f for f in os.listdir(images_dir)
    if f.lower().endswith((".png",".jpg",".jpeg"))
    and os.path.exists(os.path.join(masks_dir, f))
])

print(f"Found {len(files)} pairs")

ds = RootDataset(images_dir, masks_dir, files, img_size=IMG_SIZE, train=True)

for i in range(min(3, len(ds))):
    img, mask = ds[i]
    print(f"Sample {i}: image={tuple(img.shape)}  mask={tuple(mask.shape)}  root_px={int(mask.sum())}")

print("All samples loaded successfully - safe to train")

Found 115 pairs
Sample 0: image=(3, 768, 768)  mask=(1, 768, 768)  root_px=941
Sample 1: image=(3, 768, 768)  mask=(1, 768, 768)  root_px=1153
Sample 2: image=(3, 768, 768)  mask=(1, 768, 768)  root_px=1828
All samples loaded successfully - safe to train


In [28]:
import os, gc, torch, sys

# Remove old checkpoint — it learned the wrong thing and must be discarded
ckpt = '/content/root_project/checkpoints/best_model.pth'
if os.path.exists(ckpt): os.remove(ckpt); print("Old checkpoint deleted")

gc.collect(); torch.cuda.empty_cache()

# Clear module cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train','dataset','model','loss']):
        del sys.modules[mod]

from train import train

train(
    images_dir      = "/content/root_project/data/images",
    masks_dir       = "/content/root_project/data/masks",
    checkpoint_path = ckpt,
    resume          = False,
    epochs          = 100,
    batch_size      = 2,
    lr              = 5e-5,
    img_size        = 768,
    threshold       = 0.5,
    patience        = 20,
)

Old checkpoint deleted
Found 115 pairs
Train: 97 | Val: 18
Device: cuda


/content/root_project/src/train.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


Epoch 001 | Train 1.1820 | Val 1.0963 | Dice 0.0303 | Recall 0.3364 | LR 5.00e-06
  -> saved (dice=0.0303  recall=0.3364)
Epoch 002 | Train 1.1381 | Val 1.0709 | Dice 0.0368 | Recall 0.4158 | LR 1.00e-05
  -> saved (dice=0.0368  recall=0.4158)
Epoch 003 | Train 1.0588 | Val 1.0060 | Dice 0.0580 | Recall 0.6229 | LR 1.50e-05
  -> saved (dice=0.0580  recall=0.6229)
Epoch 004 | Train 0.9917 | Val 0.9386 | Dice 0.0815 | Recall 0.7706 | LR 2.00e-05
  -> saved (dice=0.0815  recall=0.7706)
Epoch 005 | Train 0.9508 | Val 0.8865 | Dice 0.0957 | Recall 0.8354 | LR 2.50e-05
  -> saved (dice=0.0957  recall=0.8354)
Epoch 006 | Train 0.9207 | Val 0.8602 | Dice 0.1031 | Recall 0.8498 | LR 3.00e-05
  -> saved (dice=0.1031  recall=0.8498)
Epoch 007 | Train 0.8975 | Val 0.8379 | Dice 0.0984 | Recall 0.9070 | LR 3.50e-05
Epoch 008 | Train 0.8764 | Val 0.8256 | Dice 0.0980 | Recall 0.9137 | LR 4.00e-05
Epoch 009 | Train 0.8540 | Val 0.8137 | Dice 0.0990 | Recall 0.9258 | LR 4.50e-05
Epoch 010 | Train 0.83

In [ ]:
from google.colab import files
files.download('/content/root_project/checkpoints/best_model.pth')

In [29]:
from google.colab import files
print('Upload test images...')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'/content/root_project/data/test_images/{fname}', 'wb') as f:
        f.write(data)
print(f"Test images: {len([f for f in os.listdir('/content/root_project/data/test_images')])}")

Upload test images...


Test images: 2


In [30]:
exec(open('/content/root_project/src/predict.py').read())

Device: cuda
Model loaded from: /content/root_project/checkpoints/best_model.pth

Processing: 92-1.jpg
  Prob map: min=0.000 max=0.998 mean=0.0484
  Threshold=0.5: 410798 root px (4.353%)
  TRL = 12053.1 px = 1205.31 mm = 120.53 cm
  PRL = 1758.9 px = 175.89 mm = 17.59 cm

Processing: 92-2.jpg
  Prob map: min=0.000 max=0.990 mean=0.1200
  Threshold=0.5: 1085753 root px (11.505%)
  TRL = 35515.0 px = 3551.50 mm = 355.15 cm
  PRL = 2048.9 px = 204.89 mm = 20.49 cm

image  TRL_pixels  PRL_pixels  TRL_cm  PRL_cm  root_coverage_pct
 92-1    12053.13     1758.88  120.53   17.59             4.3530
 92-2    35515.05     2048.90  355.15   20.49            11.5051

Saved: /content/root_project/outputs/results/results.csv


In [31]:
files.download('/content/root_project/outputs/results/results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>